# 99 — Teardown

Deletes every resource created by this demo so it stops costing money.

**Leaves in place:** the notebooks themselves, the generator, the architecture diagram. Everything can be re-spun by running `00_setup` again.

In [0]:
%pip install -q databricks-sdk --upgrade
dbutils.library.restartPython()

In [0]:
LAKEBASE_INSTANCE_NAME = "mccain-cold-chain-sju"
UC_CATALOG = "main"
UC_SCHEMA = "mccain_cold_chain_sju"
MODEL_NAME = f"{UC_CATALOG}.{UC_SCHEMA}.cold_chain_anomaly_detector"

# Safety toggle — set to True before running to actually tear things down.
CONFIRM_DESTROY = False

In [0]:
assert CONFIRM_DESTROY, "Set CONFIRM_DESTROY = True in the cell above, then re-run."

In [0]:
# Drop the demo schema (all silver/gold/bronze tables go with it)
spark.sql(f"DROP SCHEMA IF EXISTS {UC_CATALOG}.{UC_SCHEMA} CASCADE")
print(f"  ✅ dropped {UC_CATALOG}.{UC_SCHEMA}")

In [0]:
import mlflow
mlflow.set_registry_uri("databricks-uc")
try:
    client = mlflow.MlflowClient()
    client.delete_registered_model(MODEL_NAME)
    print(f"  ✅ deleted model {MODEL_NAME}")
except Exception as e:
    print(f"  ⚠️  {MODEL_NAME} — {e}")

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
try:
    inst = w.database.get_database_instance(name=LAKEBASE_INSTANCE_NAME)
    w.database.delete_database_instance(name=LAKEBASE_INSTANCE_NAME, purge=True)
    print(f"  ✅ deleted Lakebase instance {LAKEBASE_INSTANCE_NAME}")
except Exception as e:
    print(f"  ⚠️  {LAKEBASE_INSTANCE_NAME} — {e}")

## Done. Run `00_setup` any time to rebuild the whole demo.